In [ ]:
# ======================================================
# Notebook: Bayesian-style optimisation using Neural Network
# Select next (10,2) inputs for noisy black-box maximisation
# ======================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Load data
X = np.load("/mnt/data/initial_inputs.npy")     # (N,2)
y = np.load("/mnt/data/initial_outputs.npy")    # (N,)

# Normalize targets (stabilizes training)
y_norm = (y - y.mean()) / (y.std() + 1e-8)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y_norm.reshape(-1,1), dtype=torch.float32)

# Neural network
model = nn.Sequential(
    nn.Linear(2, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 1)
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train
model.train()
for _ in range(600):
    optimizer.zero_grad()
    pred = model(X_t)
    loss = criterion(pred, y_t)
    loss.backward()
    optimizer.step()

# Candidate grid
x_min, x_max = X[:,0].min(), X[:,0].max()
y_min, y_max = X[:,1].min(), X[:,1].max()

gx = np.linspace(x_min, x_max, 100)
gy = np.linspace(y_min, y_max, 100)
XX, YY = np.meshgrid(gx, gy)
X_grid = np.vstack([XX.ravel(), YY.ravel()]).T
X_grid_t = torch.tensor(X_grid, dtype=torch.float32)

# MC Dropout for uncertainty
model.train()
samples = []

with torch.no_grad():
    for _ in range(30):
        samples.append(model(X_grid_t).numpy())

samples = np.stack(samples)
mean_pred = samples.mean(axis=0).flatten()
uncertainty = samples.std(axis=0).flatten()

# Acquisition = exploitation + exploration
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,2)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,2) inputs:")
print(next_points)

# Plot
plt.figure(figsize=(6,5))
plt.contourf(XX, YY, mean_pred.reshape(XX.shape), levels=30)
plt.scatter(X[:,0], X[:,1])
plt.scatter(next_points[:,0], next_points[:,1], marker='x', s=100)
plt.show()